# 05 — Feature: CPU
Parses the `Cpu` string into structured features and applies target encoding.

**Output features:** `CPU_GHz`, `CPU_Brand_TE`, `CPU_Family_TE`, `CPU_TE`

In [ ]:
!pip install category_encoders

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import re
import category_encoders as ce

In [ ]:
ld = pd.read_csv("laptop_data_features.csv")
print(f"Shape: {ld.shape}")

In [ ]:
# Explore Cpu
print(ld['Cpu'].value_counts().head(15))
print(f"\nUnique CPUs: {ld['Cpu'].nunique()}")

In [ ]:
# Parse CPU string into Brand, Family, GHz
def parse_cpu(cpu_str):
    if pd.isna(cpu_str):
        return pd.Series(['Unknown', 'Unknown', np.nan],
                         index=['CPU_Brand', 'CPU_Family', 'CPU_GHz'])
    tokens = cpu_str.split()
    brand  = tokens[0]                               # Intel, AMD, Samsung
    family = tokens[1] if len(tokens) > 1 else 'Unknown'  # Core, Celeron, A-Series
    ghz_match = re.search(r'(\d+\.?\d*)\s?GHz', cpu_str)
    ghz = float(ghz_match.group(1)) if ghz_match else np.nan
    return pd.Series([brand, family, ghz],
                     index=['CPU_Brand', 'CPU_Family', 'CPU_GHz'])

ld[['CPU_Brand', 'CPU_Family', 'CPU_GHz']] = ld['Cpu'].apply(parse_cpu)
print("Parse sample:")
print(ld[['Cpu', 'CPU_Brand', 'CPU_Family', 'CPU_GHz']].head(5).to_string())

In [ ]:
# Handle missing GHz
print(f"Missing CPU_GHz: {ld['CPU_GHz'].isnull().sum()}")
ld['CPU_GHz'].fillna(ld['CPU_GHz'].median(), inplace=True)

In [ ]:
# Visualize Brand vs Price
plt.figure(figsize=(8, 4))
brand_mean = ld.groupby('CPU_Brand')['Price'].mean().sort_values(ascending=False)
brand_mean.plot(kind='bar', color='coral')
plt.title("Mean Price by CPU Brand")
plt.xlabel("CPU Brand")
plt.ylabel("Mean Price (INR)")
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

In [ ]:
# Visualize top CPU Families vs Price
plt.figure(figsize=(14, 4))
family_mean = ld.groupby('CPU_Family')['Price'].mean().sort_values(ascending=False).head(12)
family_mean.plot(kind='bar', color='mediumseagreen')
plt.title("Top CPU Families by Mean Price")
plt.xlabel("CPU Family")
plt.ylabel("Mean Price (INR)")
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

In [ ]:
# CPU GHz vs Price scatter
plt.figure(figsize=(7, 4))
plt.scatter(ld['CPU_GHz'], ld['Price'], alpha=0.4, color='steelblue')
plt.title("CPU GHz vs Price")
plt.xlabel("GHz")
plt.ylabel("Price (INR)")
plt.tight_layout()
plt.show()
print(f"Correlation CPU_GHz vs Price: {ld['CPU_GHz'].corr(ld['Price']):.4f}")

In [ ]:
# Target mean encode Brand
brand_mean = ld.groupby('CPU_Brand')['Price'].mean()
ld['CPU_Brand_TE'] = ld['CPU_Brand'].map(brand_mean)

# Target mean encode Family
family_mean = ld.groupby('CPU_Family')['Price'].mean()
ld['CPU_Family_TE'] = ld['CPU_Family'].map(family_mean)

In [ ]:
# Target encode full CPU string (captures model-level detail)
encoder = ce.TargetEncoder(cols=['Cpu'])
ld['CPU_TE'] = encoder.fit_transform(ld[['Cpu']], ld['Price'])

In [ ]:
# Correlation with Price
cpu_cols = ['CPU_GHz', 'CPU_Brand_TE', 'CPU_Family_TE', 'CPU_TE']
print("Correlation with Price:")
print(ld[cpu_cols + ['Price']].corr()['Price'].sort_values(ascending=False))

In [ ]:
# Drop raw and intermediate string columns
ld.drop(columns=['Cpu', 'CPU_Brand', 'CPU_Family'], inplace=True)

In [ ]:
ld.to_csv("laptop_data_features.csv", index=False)
print("Saved: laptop_data_features.csv")
print(f"Shape: {ld.shape}")
print(f"Columns: {list(ld.columns)}")